# **LIBRARIES** NEEDED


In [ ]:
# Data Handling
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# ML Tools
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# Model
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    roc_auc_score,
    roc_curve
)

# **ADDING THE FILE FROM KAGGLE**

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (1).csv


In [ ]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")
df.head()


df = df.dropna()
df.isnull().sum()
df.shape

(7043, 21)

# CHECKING CORRELATION WITH **CHURN**

In [ ]:
le = LabelEncoder()
df['Churn_Encoded'] = le.fit_transform(df['Churn'])
display(df[['Churn', 'Churn_Encoded']].head())

,Churn,Churn_Encoded
0,No,0
1,No,0
2,Yes,1
3,No,0
4,Yes,1


### Identifying numerical features and calculating correlations

 select all numerical columns (including the newly encoded 'Churn' column) and compute their correlation matrix. then focus on the correlations with 'Churn_Encoded'.

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
correlation_matrix = df[numerical_cols].corr()

# Display correlation with Churn
churn_correlations = correlation_matrix['Churn_Encoded'].sort_values(ascending=False)
display(churn_correlations)

,Churn_Encoded
Churn_Encoded,1.000000
MonthlyCharges,0.193356
SeniorCitizen,0.150889
tenure,-0.352229


### Visualizing correlations with a heatmap

A heatmap provides a clear visual representation of the correlation strength between all numerical features.

# **CHECKING THE IMPORTANT FEATURES**
 we look for churn distrubution
 convert churn to numeric
 we label and train our model to only select relevant features


In [ ]:
df['Churn'].unique()


array([nan])

In [ ]:

df['Churn'].unique()

df['Churn'].value_counts()

df['Churn'].dtype

dtype('int64')

In [ ]:
print("First 10 Churn values:")
print(df['Churn'].head(10))

print("\nUnique values:")
print(df['Churn'].unique())

print("\nValue counts:")
print(df['Churn'].value_counts())

print("\nDatatype:")
print(df['Churn'].dtype)

First 10 Churn values:
0    0
1    0
2    1
3    0
4    1
5    1
6    0
7    0
8    1
9    0
Name: Churn, dtype: int64

Unique values:
[0 1]

Value counts:
Churn
0    5174
1    1869
Name: count, dtype: int64

Datatype:
int64


In [ ]:
# Create list of columns to remove
columns_to_remove = []

# Remove customerID (useless unique identifier)
if 'customerID' in df.columns:
    columns_to_remove.append('customerID')

# Remove duplicate target if present
if 'Churn_Encoded' in df.columns:
    columns_to_remove.append('Churn_Encoded')

# Drop columns
df = df.drop(columns=columns_to_remove)

# Show result
print("Removed Columns:", columns_to_remove)

print("\nRemaining Columns:\n")
print(df.columns)

print("\nCurrent Shape:")
print(df.shape)

Removed Columns: []

Remaining Columns:

Index(['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity',
       'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV',
       'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod',
       'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

Current Shape:
(7032, 20)


In [ ]:
import numpy as np

# Replace blank spaces with NaN
df['TotalCharges'] = df['TotalCharges'].replace(" ", np.nan)

# Convert to numeric
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'])

# Remove rows with missing values
df = df.dropna()

# Verify
print("TotalCharges datatype:", df['TotalCharges'].dtype)

print("\nDataset shape after fixing TotalCharges:")
print(df.shape)

TotalCharges datatype: float64

Dataset shape after fixing TotalCharges:
(7032, 20)


In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# =====================================
# Step 1 — Convert Categorical to Numeric
# =====================================

df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

print("Encoded Dataset Shape:")
print(df_encoded.shape)

# =====================================
# Step 2 — Separate Features & Target
# =====================================

X = df_encoded.drop('Churn', axis=1)

y = df_encoded['Churn']

# =====================================
# Step 3 — Scale Features
# =====================================

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# =====================================
# Step 4 — Train Logistic Regression
# =====================================

model = LogisticRegression(max_iter=1000)

model.fit(X_scaled, y)

# =====================================
# Step 5 — Compare Every Column With Churn
# (Feature Importance)
# =====================================

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_[0]
})

# Absolute importance
feature_importance['Impact'] = abs(
    feature_importance['Coefficient']
)

# Sort strongest first
feature_importance = feature_importance.sort_values(
    by='Impact',
    ascending=False
)

print("\nTop Features Affecting Churn:\n")

print(feature_importance.head(15))

# =====================================
# Step 6 — Show Features That Affect Churn Most
# =====================================

print("\nMost Influential Churn Drivers:\n")

strong_features = feature_importance.head(10)

print(strong_features)

# =====================================
# Step 7 — Show Weak Features
# =====================================

threshold = 0.05

weak_features = feature_importance[
    feature_importance['Impact'] < threshold
]['Feature']

print("\nNumber of Weak Features:")

print(len(weak_features))

print("\nSample Weak Features:")

print(weak_features.tolist()[:10])


Encoded Dataset Shape:
(7032, 31)

Top Features Affecting Churn:

                           Feature  Coefficient    Impact
1                           tenure    -1.417870  1.417870
3                     TotalCharges     0.671572  0.671572
25               Contract_Two year    -0.572107  0.572107
10     InternetService_Fiber optic     0.561906  0.561906
2                   MonthlyCharges    -0.456669  0.456669
24               Contract_One year    -0.268698  0.268698
23             StreamingMovies_Yes     0.172778  0.172778
26            PaperlessBilling_Yes     0.168484  0.168484
21                 StreamingTV_Yes     0.168256  0.168256
9                MultipleLines_Yes     0.160835  0.160835
13              OnlineSecurity_Yes    -0.148155  0.148155
28  PaymentMethod_Electronic check     0.143443  0.143443
19                 TechSupport_Yes    -0.137689  0.137689
0                    SeniorCitizen     0.079767  0.079767
6                   Dependents_Yes    -0.068719  0.068719

Most 

In [ ]:
# =====================================
# Step 1 — Define Threshold
# =====================================

threshold = 0.05   # Features below this impact will be removed

# =====================================
# Step 2 — Identify Weak Features
# =====================================

weak_features = feature_importance[
    feature_importance['Impact'] < threshold
]['Feature']

print("Number of Weak Features Removed:")

print(len(weak_features))

print("\nSample Weak Features Removed:")

print(weak_features.tolist()[:15])

# =====================================
# Step 3 — Remove Weak Features
# =====================================

X_reduced = X.drop(columns=weak_features)

print("\nFinal Dataset Shape After Feature Removal:")

print(X_reduced.shape)

# =====================================
# Step 4 — Prepare Final Dataset
# =====================================

# Combine final features with target

final_df = X_reduced.copy()

final_df['Churn'] = y

print("\nFinal Dataset Ready for Modeling:")

print(final_df.shape)

print("\nFinal Columns:\n")

print(final_df.columns)
df.head()

Number of Weak Features Removed:
8

Sample Weak Features Removed:
['PhoneService_Yes', 'MultipleLines_No phone service', 'OnlineBackup_Yes', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Mailed check', 'DeviceProtection_Yes', 'gender_Male', 'Partner_Yes']

Final Dataset Shape After Feature Removal:
(7032, 22)

Final Dataset Ready for Modeling:
(7032, 23)

Final Columns:

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges',
       'Dependents_Yes', 'MultipleLines_Yes', 'InternetService_Fiber optic',
       'InternetService_No', 'OnlineSecurity_No internet service',
       'OnlineSecurity_Yes', 'OnlineBackup_No internet service',
       'DeviceProtection_No internet service',
       'TechSupport_No internet service', 'TechSupport_Yes',
       'StreamingTV_No internet service', 'StreamingTV_Yes',
       'StreamingMovies_No internet service', 'StreamingMovies_Yes',
       'Contract_One year', 'Contract_Two year', 'PaperlessBilling_Yes',
       'PaymentMethod_Electr

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,0
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,0
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,0
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1


# **MODEL BUILDING **

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report,
    roc_auc_score
)

# =====================================
# Step 1 — Split Features & Target
# =====================================

X_final = final_df.drop('Churn', axis=1)

y_final = final_df['Churn']

# =====================================
# Step 2 — Train-Test Split
# =====================================

X_train, X_test, y_train, y_test = train_test_split(
    X_final,
    y_final,
    test_size=0.2,
    random_state=42
)

# =====================================
# Step 3 — Scale Features
# =====================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

# =====================================
# Step 4 — Train Final Model
# =====================================

model = LogisticRegression(max_iter=1000)

model.fit(
    X_train_scaled,
    y_train
)

# =====================================
# Step 5 — Predict Churn
# =====================================

y_pred = model.predict(X_test_scaled)

# =====================================
# Step 6 — Model Evaluation
# =====================================

print("Final Accuracy:")

print(
    accuracy_score(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nROC-AUC Score:")

print(
    roc_auc_score(
        y_test,
        y_pred
    )
)

Final Accuracy:
0.7910447761194029

Confusion Matrix:
[[919 114]
 [180 194]]

Classification Report:
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.63      0.52      0.57       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.72      1407
weighted avg       0.78      0.79      0.78      1407


ROC-AUC Score:
0.7041791987410118
